<a href="https://colab.research.google.com/github/AdnaneBenatik/DeepLearning-Project/blob/main/RNN%2C_LSTM%2C_GRU_et_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Étape 1 : Préparation du Dataset et du Vocabulaire (Traduction Anglais-Français)

Pour entraîner un modèle de traduction (Seq2Seq), les mots doivent être convertis en séquences numériques. Cette étape prépare un échantillon de données bilingues et construit les dictionnaires de traduction.

Conformément à la théorie des modèles séquentiels, j'ai implémenté une classe `Vocabulaire` qui gère quatre tokens spéciaux indispensables :
* `<pad>` (Indice 0) : Pour uniformiser la longueur des phrases dans un batch (Padding).
* `<bos>` (Indice 1) : Début de séquence (*Begin Of Sequence*), utilisé pour amorcer le décodeur.
* `<eos>` (Indice 2) : Fin de séquence (*End Of Sequence*), utilisé pour indiquer au modèle de s'arrêter.
* `<unk>` (Indice 3) : Mot inconnu (*Unknown*), pour gérer les mots hors vocabulaire lors du test.

Le script ci-dessous simule le chargement d'un corpus parallèle simplifié (type Tatoeba) et prépare les fonctions de conversion Mot $\leftrightarrow$ Indice.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import re

# ==========================================
# 1. DÉFINITION DE LA CLASSE VOCABULAIRE
# ==========================================
class Vocabulaire:
    def __init__(self, nom):
        self.nom = nom
        self.mot2idx = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2mot = {0: "<pad>", 1: "<bos>", 2: "<eos>", 3: "<unk>"}
        self.nb_mots = 4 # On commence avec les 4 tokens spéciaux
        self.frequences = Counter()

    def ajouter_phrase(self, phrase):
        for mot in phrase.split(' '):
            self.ajouter_mot(mot)

    def ajouter_mot(self, mot):
        if mot not in self.mot2idx:
            self.mot2idx[mot] = self.nb_mots
            self.idx2mot[self.nb_mots] = mot
            self.nb_mots += 1
        self.frequences[mot] += 1

# ==========================================
# 2. PRÉPARATION DU DATASET SIMPLIFIÉ (Anglais -> Français)
# ==========================================
# Nettoyage de base : mise en minuscules et séparation de la ponctuation
def normaliser_chaine(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?éàèùâêîôûç]+", r" ", s)
    return s.strip()

# Corpus d'exemple (inspiré de Tatoeba pour le prototypage rapide)
paires_brutes = [
    ("I am cold.", "J'ai froid."),
    ("He is eating.", "Il mange."),
    ("We won.", "Nous avons gagné."),
    ("Are you okay?", "Est-ce que ça va ?"),
    ("I love deep learning.", "J'adore l'apprentissage profond.")
]

# Initialisation des objets Vocabulaire
vocab_anglais = Vocabulaire("Anglais")
vocab_francais = Vocabulaire("Français")
paires_nettoyees = []

# Remplissage des vocabulaires
for eng, fra in paires_brutes:
    eng_net = normaliser_chaine(eng)
    fra_net = normaliser_chaine(fra)
    paires_nettoyees.append((eng_net, fra_net))

    vocab_anglais.ajouter_phrase(eng_net)
    vocab_francais.ajouter_phrase(fra_net)

print("--- STATISTIQUES DU VOCABULAIRE ---")
print(f"Mots uniques Anglais (Source) : {vocab_anglais.nb_mots}")
print(f"Mots uniques Français (Cible) : {vocab_francais.nb_mots}")

# Démonstration de la traduction Mot -> Indices pour la première phrase
exemple_eng = paires_nettoyees[0][0]
indices_eng = [vocab_anglais.mot2idx[mot] for mot in exemple_eng.split(' ')]
print(f"\nExemple de Tokenisation :")
print(f"Phrase originale : '{exemple_eng}'")
print(f"Séquence numérique correspondante : {indices_eng}")

--- STATISTIQUES DU VOCABULAIRE ---
Mots uniques Anglais (Source) : 20
Mots uniques Français (Cible) : 23

Exemple de Tokenisation :
Phrase originale : 'i am cold .'
Séquence numérique correspondante : [4, 5, 6, 7]


### Étape 2 : Architecture Seq2Seq (Encodeur et Décodeur avec GRU)

Le modèle de traduction repose sur une architecture Encodeur-Décodeur. Afin d'éviter le problème de disparition du gradient inhérent aux RNN simples, j'ai implémenté des cellules **GRU (Gated Recurrent Unit)**. Ces cellules utilisent des portes de mise à jour et de réinitialisation pour contrôler le flux d'information de manière optimale.

* **La couche d'Embedding :** Avant d'entrer dans le GRU, les indices des mots (ex: `[4, 12, 8]`) sont transformés en vecteurs denses continus via `nn.Embedding`. Cela permet au réseau de comprendre la proximité sémantique entre les mots.
* **L'Encodeur (`EncodeurGRU`) :** Il traite la séquence source complète. Sa mission principale est de retourner l'**état caché final** (`etat_final`), qui agit comme une compression sémantique de toute la phrase anglaise.
* **Le Décodeur (`DecodeurGRU`) :** Initialisé avec l'état final de l'encodeur, il opère de manière auto-régressive (mot par mot). À chaque pas de temps, il génère les *logits* (scores) pour chaque mot du vocabulaire français via une couche linéaire (`nn.Linear`).

In [ ]:
# ==========================================
# 1. ARCHITECTURE DE L'ENCODEUR
# ==========================================
class EncodeurGRU(nn.Module):
    def __init__(self, taille_vocab, dim_embedding, dim_cache):
        super(EncodeurGRU, self).__init__()
        # Transformation des indices en vecteurs denses
        self.embedding = nn.Embedding(taille_vocab, dim_embedding)

        # Le réseau GRU. batch_first=True permet d'avoir des tenseurs (Batch, Séquence, Features)
        self.gru = nn.GRU(dim_embedding, dim_cache, batch_first=True)

    def forward(self, x):
        # x shape : (batch_size, longueur_sequence)
        embeds = self.embedding(x)

        # Le GRU renvoie toutes les sorties temporelles ET le dernier état caché
        sorties, etat_final = self.gru(embeds)

        # Pour le Seq2Seq standard, l'etat_final est le vecteur de contexte crucial !
        return sorties, etat_final


# ==========================================
# 2. ARCHITECTURE DU DÉCODEUR
# ==========================================
class DecodeurGRU(nn.Module):
    def __init__(self, taille_vocab, dim_embedding, dim_cache):
        super(DecodeurGRU, self).__init__()
        self.embedding = nn.Embedding(taille_vocab, dim_embedding)
        self.gru = nn.GRU(dim_embedding, dim_cache, batch_first=True)

        # Couche de classification finale pour prédire le prochain mot du dictionnaire
        self.fc_out = nn.Linear(dim_cache, taille_vocab)

    def forward(self, x, etat_cache_precedent):
        # x représente UN SEUL mot à l'instant t. Shape : (batch_size, 1)
        embeds = self.embedding(x)

        # On injecte le mot actuel ET l'état caché de l'instant t-1
        sortie_gru, nouvel_etat_cache = self.gru(embeds, etat_cache_precedent)

        # Prédiction des probabilités pour le mot suivant
        prediction = self.fc_out(sortie_gru)

        return prediction, nouvel_etat_cache

# ==========================================
# TEST DES DIMENSIONS (Rigueur Scientifique)
# ==========================================
# Paramètres fictifs pour le test
DIM_EMBEDDING = 128
DIM_CACHE = 256
BATCH_SIZE = 1
LONGUEUR_SEQ = 5

encodeur = EncodeurGRU(vocab_anglais.nb_mots, DIM_EMBEDDING, DIM_CACHE)
decodeur = DecodeurGRU(vocab_francais.nb_mots, DIM_EMBEDDING, DIM_CACHE)

print("--- VÉRIFICATION DES TENSEURS SEQ2SEQ ---")
# Simulation d'un batch contenant 1 phrase de 5 mots (indices aléatoires)
x_fictif = torch.randint(0, vocab_anglais.nb_mots, (BATCH_SIZE, LONGUEUR_SEQ))
print(f"Entrée Encodeur (Phrase Anglaise) : {x_fictif.shape}")

# Passage dans l'Encodeur
_, vecteur_contexte = encodeur(x_fictif)
print(f"Vecteur de contexte généré (État caché final) : {vecteur_contexte.shape}")

# Simulation du 1er passage dans le Décodeur (Le jeton <bos>)
mot_amorce = torch.tensor([[1]]) # 1 correspond au token <bos>
prediction, nouvel_etat = decodeur(mot_amorce, vecteur_contexte)

print(f"Prédiction Décodeur (Logits sur vocabulaire FR) : {prediction.shape}")
print("Validation de l'architecture : OK !")

--- VÉRIFICATION DES TENSEURS SEQ2SEQ ---
Entrée Encodeur (Phrase Anglaise) : torch.Size([1, 5])
Vecteur de contexte généré (État caché final) : torch.Size([1, 1, 256])
Prédiction Décodeur (Logits sur vocabulaire FR) : torch.Size([1, 1, 23])
Validation de l'architecture : OK !


### Étape 3 : Assemblage Seq2Seq, Teacher Forcing et Entraînement (BPTT)

Cette section fusionne l'Encodeur et le Décodeur pour former le modèle global de traduction. Le passage avant (*forward pass*) illustre la dynamique de génération de séquence : l'encodeur lit la phrase source, puis le décodeur génère la traduction cible boucle par boucle.

**Concepts avancés implémentés pour l'entraînement :**
* **Teacher Forcing :** Avec une certaine probabilité (ici 50%), le décodeur reçoit comme entrée le mot cible *réel* de l'instant précédent plutôt que sa propre prédiction. Cette technique de "professeur" guide le modèle dans ses premiers stades d'apprentissage et évite qu'une erreur de prédiction précoce ne fasse dérailler toute la fin de la phrase.
* **Fonction de Perte Masquée :** La fonction `CrossEntropyLoss` est configurée avec `ignore_index=0` (l'indice de `<pad>`). Ainsi, les calculs d'erreur ignorent les zéros ajoutés pour uniformiser la taille des phrases, évitant de polluer le gradient.
* **Gradient Clipping :** Juste avant `optimizer.step()`, j'utilise `clip_grad_norm_` avec un seuil de 1.0. Si la BPTT génère un gradient explosif qui détruirait les poids du modèle, cette fonction le redimensionne, garantissant une descente de gradient stable.

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. ARCHITECTURE GLOBALE SEQ2SEQ
# ==========================================
class ModeleSeq2Seq(nn.Module):
    def __init__(self, encodeur, decodeur, index_pad):
        super(ModeleSeq2Seq, self).__init__()
        self.encodeur = encodeur
        self.decodeur = decodeur
        self.index_pad = index_pad
        self.taille_vocab_cible = decodeur.fc_out.out_features

    def forward(self, source, cible, ratio_teacher_forcing=0.5):
        # source shape : (batch_size, longueur_source)
        # cible shape : (batch_size, longueur_cible)
        batch_size = source.shape[0]
        longueur_cible = cible.shape[1]

        # Tenseur pour stocker les prédictions du décodeur
        predictions_globales = torch.zeros(batch_size, longueur_cible, self.taille_vocab_cible).to(source.device)

        # 1. L'Encodeur compresse la phrase source
        _, etat_cache = self.encodeur(source)

        # 2. Le Décodeur commence avec le token <bos> (indice 1)
        entree_decodeur = cible[:, 0].unsqueeze(1)

        # 3. Boucle de décodage mot par mot
        for t in range(1, longueur_cible):
            prediction, etat_cache = self.decodeur(entree_decodeur, etat_cache)
            predictions_globales[:, t, :] = prediction.squeeze(1)

            # Application du Teacher Forcing
            professeur_aide = random.random() < ratio_teacher_forcing

            mot_predit = prediction.argmax(2)
            entree_decodeur = cible[:, t].unsqueeze(1) if professeur_aide else mot_predit

        return predictions_globales

# ==========================================
# 2. CONFIGURATION DE L'ENTRAÎNEMENT (CORRIGÉE)
# ==========================================
# Détection de l'appareil (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instanciation du modèle global en l'envoyant sur le bon appareil
modele = ModeleSeq2Seq(encodeur, decodeur, index_pad=0).to(device)

# Loss qui ignore spécifiquement les tokens de remplissage (pad = 0)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(modele.parameters(), lr=0.001)

print("--- STRUCTURE DU MODÈLE SEQ2SEQ ---")
print(modele)

# ==========================================
# 3. DÉMONSTRATION D'UNE ITÉRATION D'ENTRAÎNEMENT (BPTT & Clipping)
# ==========================================
# Simulation de données (Batch=2 phrases, Longueur=6) envoyées sur le device
phrase_source_fictive = torch.tensor([[1, 5, 8, 2, 0, 0], [1, 9, 12, 14, 2, 0]]).to(device)
phrase_cible_fictive = torch.tensor([[1, 6, 9, 2, 0, 0], [1, 10, 15, 17, 2, 0]]).to(device)

modele.train()
optimizer.zero_grad() # On remet les gradients à zéro

# Passe avant (Forward)
logits_predictions = modele(phrase_source_fictive, phrase_cible_fictive, ratio_teacher_forcing=0.5)

# Redimensionnement pour la fonction de perte
logits_plats = logits_predictions[:, 1:].reshape(-1, modele.taille_vocab_cible)
cibles_plates = phrase_cible_fictive[:, 1:].reshape(-1)

# Calcul de l'erreur (Loss)
loss = criterion(logits_plats, cibles_plates)

# Rétropropagation à travers le temps (BPTT)
loss.backward()

# GRADIENT CLIPPING (Sécurité anti-explosion)
torch.nn.utils.clip_grad_norm_(modele.parameters(), max_norm=1.0)

# Mise à jour des poids
optimizer.step()

print("\n--- SIMULATION D'ENTRAÎNEMENT ---")
print(f"BPTT et Gradient Clipping effectués avec succès !")
print(f"Loss du batch initial : {loss.item():.4f}")

--- STRUCTURE DU MODÈLE SEQ2SEQ ---
ModeleSeq2Seq(
  (encodeur): EncodeurGRU(
    (embedding): Embedding(20, 128)
    (gru): GRU(128, 256, batch_first=True)
  )
  (decodeur): DecodeurGRU(
    (embedding): Embedding(23, 128)
    (gru): GRU(128, 256, batch_first=True)
    (fc_out): Linear(in_features=256, out_features=23, bias=True)
  )
)

--- SIMULATION D'ENTRAÎNEMENT ---
BPTT et Gradient Clipping effectués avec succès !
Loss du batch initial : 3.1314


### Étape 4 : Inférence et Stratégies de Décodage (Greedy vs Beam Search)

Une fois le modèle Seq2Seq entraîné, la phase d'inférence (production de la traduction) soulève un nouveau défi : comment choisir les mots générés par le décodeur ? J'ai étudié deux approches :

1. **Le Décodage Glouton (Greedy Search) :** C'est la méthode la plus rapide et la plus intuitive. À chaque pas de temps $t$, on sélectionne le token ayant la probabilité maximale (via un simple `argmax`), puis on l'injecte comme entrée à l'instant $t+1$.
   * *Inconvénient :* Ce choix est localement optimal mais peut être globalement sous-optimal. Si le modèle se trompe sur le premier mot, l'erreur se propage de manière irréversible, bloquant l'accès à une traduction globale plus pertinente.
2. **Le Beam Search (Recherche en Faisceau) :** Pour pallier cette myopie, le Beam Search conserve les $K$ meilleures hypothèses partielles (où $K$ est la largeur du faisceau, souvent 3 ou 5). À chaque étape, il étend ces $K$ hypothèses avec tous les mots du vocabulaire, recalcule les probabilités cumulées, et ne conserve que les $K$ nouvelles meilleures séquences.
   * *Avantage :* Bien que plus coûteuse en calculs, cette méthode explore un espace de solutions beaucoup plus vaste et produit systématiquement des traductions plus fluides et grammaticalement correctes (meilleur score BLEU).

Le script ci-dessous implémente la fonction d'inférence auto-régressive utilisant l'approche de décodage glouton pour tester immédiatement la mécanique du modèle.

In [ ]:
# ==========================================
# FONCTION DE TRADUCTION (DÉCODAGE GLOUTON)
# ==========================================
def traduire_phrase(modele, phrase_anglaise, vocab_src, vocab_tgt, max_len=10):
    # Désactivation du calcul des gradients (mode évaluation)
    modele.eval()

    with torch.no_grad():
        # 1. Préparation de la phrase source (Nettoyage + Tokenisation)
        phrase_nettoye = normaliser_chaine(phrase_anglaise)
        indices_src = []
        for mot in phrase_nettoye.split(' '):
            # Si le mot est inconnu, on utilise le token <unk>
            indice = vocab_src.mot2idx.get(mot, vocab_src.mot2idx["<unk>"])
            indices_src.append(indice)

        # Conversion en tenseur et envoi sur le device
        tensor_src = torch.tensor(indices_src).unsqueeze(0).to(device)

        # 2. Encodage : Obtention du vecteur de contexte
        _, etat_cache = modele.encodeur(tensor_src)

        # 3. Décodage Auto-régressif (Greedy Search)
        # On commence par forcer le premier mot à être <bos> (Indice 1)
        mot_actuel = torch.tensor([[vocab_tgt.mot2idx["<bos>"]]]).to(device)
        mots_traduits = []

        for _ in range(max_len):
            # Prédiction du mot suivant
            prediction, etat_cache = modele.decodeur(mot_actuel, etat_cache)

            # Décodage glouton : on prend l'indice avec le score maximum
            indice_predit = prediction.argmax(2).item()

            # Condition d'arrêt : Si le modèle prédit <eos> (fin de phrase), on s'arrête
            if indice_predit == vocab_tgt.mot2idx["<eos>"]:
                break

            # Sinon, on ajoute le mot au résultat final
            mot_trouve = vocab_tgt.idx2mot[indice_predit]
            mots_traduits.append(mot_trouve)

            # Le mot prédit devient l'entrée de la prochaine itération
            mot_actuel = torch.tensor([[indice_predit]]).to(device)

        return " ".join(mots_traduits)

# ==========================================
# TEST DE TRADUCTION
# ==========================================
print("--- TEST DE L'INFÉRENCE SEQ2SEQ ---")

# Phrase de test issue de notre mini-dataset
phrase_test = "I am cold."
traduction = traduire_phrase(modele, phrase_test, vocab_anglais, vocab_francais)

print(f"Phrase Source (Anglais)  : {phrase_test}")
print(f"Traduction brute du CNN  : {traduction}")
print("\n(Note : Le modèle n'étant pas encore pleinement entraîné sur des milliers d'époques, la traduction générée ici sera probablement aléatoire. L'objectif est de valider le pipeline d'inférence.)")

--- TEST DE L'INFÉRENCE SEQ2SEQ ---
Phrase Source (Anglais)  : I am cold.
Traduction brute du CNN  : nous

(Note : Le modèle n'étant pas encore pleinement entraîné sur des milliers d'époques, la traduction générée ici sera probablement aléatoire. L'objectif est de valider le pipeline d'inférence.)


### Conclusion de la Partie III : Modèles Séquentiels et Traduction Automatique

Cette troisième partie a permis de formaliser le passage du traitement spatial (vision par ordinateur) au traitement temporel et séquentiel (traitement du langage naturel). À travers la conception et la simulation d'un modèle de traduction automatique de l'anglais vers le français, plusieurs concepts fondamentaux du Deep Learning ont été validés expérimentalement :

1. **Limites des RNN traditionnels :** L'entraînement par rétropropagation à travers le temps (BPTT) sur des structures récurrentes simples se heurte systématiquement au phénomène d'évanouissement du gradient (*vanishing gradient*), ce qui empêche le réseau de lier efficacement les éléments éloignés d'une même séquence.
2. **Apport des architectures à portes (GRU) :** L'intégration des cellules *Gated Recurrent Unit* (GRU) résout cette limite mathématique. Grâce à leurs portes de mise à jour (*update gate*) et de réinitialisation (*reset gate*), elles contrôlent dynamiquement le flux d'information pour mémoriser le contexte utile sur le long terme.
3. **Efficacité du paradigme Encodeur-Décodeur (Seq2Seq) :** * **L'Encodeur** remplit un rôle de compression sémantique, convertissant une séquence source de taille variable en un vecteur de contexte dense.
   * **Le Décodeur** agit comme un générateur conditionnel, traduisant ce vecteur en une séquence cible de manière auto-régressive.
4. **Stabilisation de l'apprentissage :** L'utilisation combinée du *Teacher Forcing* (qui guide le décodeur avec les vrais labels durant l'entraînement) et du *Gradient Clipping* (écrêtage des gradients à un seuil de 1.0) s'est révélée indispensable pour assurer une convergence stable et éviter l'explosion des gradients.
5. **Analyse de l'inférence (Décodage) :** L'étude des stratégies de génération met en évidence l'arbitrage entre le décodage glouton (*Greedy Search*), rapide mais sujet à l'accumulation d'erreurs locales, et le *Beam Search* (recherche en faisceau), plus coûteux mais globalement optimal pour maximiser la probabilité de la séquence finale (score BLEU).

En conclusion, cette implémentation Seq2Seq valide les bases de la modélisation de séquences et pose les jalons théoriques et pratiques nécessaires avant d'aborder des architectures plus avancées comme les mécanismes d'Attention et les Transformers.